In [2]:
!pip install shap

   ---------------------------------------- 0.0/547.2 kB ? eta -:--:--
   ---------------------------------------- 547.2/547.2 kB 6.1 MB/s  0:00:00

   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   ---------------------------------------- 3/3 [shap]



In [7]:
# -*- coding: utf-8 -*-
import os
import sys
import time
import glob
import argparse
import warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Sequence, Optional
from collections import defaultdict

import numpy as np
import pandas as pd
import shap
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

from sklearn.preprocessing import Normalizer, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# ==========================================
# 1. Config (from config.py)
# ==========================================
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0") # 필요에 따라 변경
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

@dataclass
class ExperimentConfig:
    data_dir: str = r"D:\Min Kim\Experiment Results\TDoA Positioning(25.06~)\dataset\original"
    file_pattern: str = "day{}.csv"
    prev_result_csv_dir: str = r"D:\Min Kim\Experiment Results\TDoA Positioning(25.06~)"
    result_dir: str = r"D:\Min Kim\Experiment Results\TDoA Positioning(25.06~)\dataset\original"

    baseline_days: List[int] = field(default_factory=lambda: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11])
    select_days: List[int] = field(default_factory=lambda: [1, 2, 3, 5, 6, 7, 8, 9, 10, 11])

    k_list: List[int] = field(default_factory=lambda: [21, 19, 16, 13, 10, 7, 4, 1])
    n_repeats: int = 1
    test_size: float = 0.2
    global_seed: int = 42

    tree_shap_sample: int = 256
    dnn_bg_sample: int = 64
    dnn_eval_sample: int = 128
    infer_runs: int = 30

    dnn_epochs: int = 50
    dnn_batch_size: int = 512
    dnn_patience: int = 5

    ml_models: List[str] = field(default_factory=lambda: ["rf", "et", "xgb", "dt", "lgbm", "cat"])
    deep_models: List[str] = field(default_factory=lambda: ["dnn"])

    @property
    def all_models(self) -> List[str]:
        return self.ml_models + self.deep_models

    @property
    def shap_dir(self) -> str:
        return str(Path(self.result_dir) / "shap")

    def ensure_output_dirs(self) -> None:
        Path(self.result_dir).mkdir(parents=True, exist_ok=True)
        Path(self.shap_dir).mkdir(parents=True, exist_ok=True)

# ==========================================
# 2. Model Registry (from model_registry.py)
# ==========================================
def configure_tensorflow_gpu() -> None:
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print(f"[GPU] TensorFlow visible GPUs: {gpus}")
        except RuntimeError as exc:
            print(exc)
    else:
        print("[GPU] No GPU detected by TensorFlow.")

def get_tf_device() -> str:
    return "/GPU:0" if tf.config.list_logical_devices("GPU") else "/CPU:0"

def set_global_seed(seed: int) -> None:
    np.random.seed(seed)
    tf.random.set_seed(seed)

def make_ml_models(seed: int = 42) -> Dict[str, object]:
    return {
        "rf": RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1),
        "et": ExtraTreesClassifier(n_estimators=200, random_state=seed, n_jobs=-1),
        "xgb": XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=seed, n_jobs=-1, tree_method="hist", device="cuda"),
        "dt": DecisionTreeClassifier(criterion="entropy", max_features="sqrt", random_state=seed),
        "lgbm": LGBMClassifier(n_estimators=300, random_state=seed, n_jobs=-1, device_type="gpu"),
        "cat": CatBoostClassifier(iterations=500, verbose=False, random_state=seed, task_type="GPU")
    }

def make_dnn(input_dim: int, num_classes: int) -> Sequential:
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(2048, activation="relu", kernel_regularizer=l2(1e-4)), BatchNormalization(), Dropout(0.1),
        Dense(1024, activation="relu", kernel_regularizer=l2(1e-4)), BatchNormalization(),
        Dense(512, activation="relu", kernel_regularizer=l2(1e-4)), BatchNormalization(),
        Dense(256, activation="relu", kernel_regularizer=l2(1e-4)), BatchNormalization(),
        Dense(num_classes, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def train_ml_model(model_name: str, x_train: pd.DataFrame, y_train: pd.Series, seed: int):
    models = make_ml_models(seed)
    model = models[model_name]
    model.fit(x_train, y_train.values.ravel())
    return model

def train_dnn_model(x_train: pd.DataFrame, y_train: pd.Series, num_classes: int, seed: int, cfg: ExperimentConfig):
    set_global_seed(seed)
    with tf.device(get_tf_device()):
        model = make_dnn(x_train.shape[1], num_classes)
        es = EarlyStopping(monitor="loss", patience=cfg.dnn_patience, restore_best_weights=True)
        model.fit(x_train.to_numpy(dtype=np.float32), y_train.to_numpy(), epochs=cfg.dnn_epochs, batch_size=cfg.dnn_batch_size, verbose=0, callbacks=[es])
    return model

def count_params_ml(model, model_name: str) -> int:
    if model_name == "dt": return int(model.tree_.node_count)
    if model_name in ["rf", "et"]: return int(sum(e.tree_.node_count for e in model.estimators_))
    if model_name == "xgb": return int(len(model.get_booster().trees_to_dataframe()))
    if model_name == "lgbm":
        dump = model.booster_.dump_model()
        def count_nodes(node):
            return 1 + count_nodes(node["left_child"]) + count_nodes(node["right_child"]) if "left_child" in node else 1
        return int(sum(count_nodes(t["tree_structure"]) for t in dump["tree_info"]))
    if model_name == "cat": return int(sum(2 * lc - 1 for lc in model.get_tree_leaf_counts()))
    return 0

def measure_inference_time(model, x_test, model_name: str, cfg: ExperimentConfig):
    x_np = x_test.to_numpy(dtype=np.float32) if isinstance(x_test, pd.DataFrame) else np.asarray(x_test, dtype=np.float32)
    warmup_fn = lambda: model.predict(x_np, verbose=0, batch_size=1024) if model_name == "dnn" else model.predict(x_np)
    warmup_fn()
    start = time.perf_counter()
    for _ in range(cfg.infer_runs): warmup_fn()
    end = time.perf_counter()
    avg_total_ms = (end - start) * 1000.0 / cfg.infer_runs
    return float(avg_total_ms), float(avg_total_ms / len(x_np))

# ==========================================
# 3. Data Utils (from data_utils.py)
# ==========================================
def build_global_label_encoder(days: Sequence[int], cfg: ExperimentConfig) -> LabelEncoder:
    all_labels = []
    for day in days:
        path = os.path.join(cfg.data_dir, cfg.file_pattern.format(day))
        df = pd.read_csv(path).dropna()
        all_labels.extend(df["label"].astype(str).tolist())
    le = LabelEncoder()
    le.fit(all_labels)
    return le

def load_split_data(days: Sequence[int], seed: int, label_encoder: LabelEncoder, cfg: ExperimentConfig):
    xt_l, xte_l, yt_l, yte_l = [], [], [], []
    for day in days:
        path = os.path.join(cfg.data_dir, cfg.file_pattern.format(day))
        df = pd.read_csv(path).dropna()
        x = df.drop(columns=["idx", "label"], errors="ignore")
        y = label_encoder.transform(df["label"].astype(str))
        x_scaled = Normalizer().fit_transform(x)
        x_df = pd.DataFrame(x_scaled, columns=x.columns)
        xt, xte, yt, yte = train_test_split(x_df, y, test_size=cfg.test_size, random_state=seed, stratify=y)
        xt_l.append(xt.reset_index(drop=True)); xte_l.append(xte.reset_index(drop=True))
        yt_l.append(pd.Series(yt).reset_index(drop=True)); yte_l.append(pd.Series(yte).reset_index(drop=True))
    return xt_l, xte_l, yt_l, yte_l

def concat_selected_features(xt_l, xte_l, yt_l, yte_l, selected_features: Sequence[str]):
    x_train = pd.concat([x[selected_features] for x in xt_l], axis=0).reset_index(drop=True)
    x_test = pd.concat([x[selected_features] for x in xte_l], axis=0).reset_index(drop=True)
    y_train = pd.concat(yt_l, axis=0).reset_index(drop=True)
    y_test = pd.concat(yte_l, axis=0).reset_index(drop=True)
    return x_train, x_test, y_train, y_test

# ==========================================
# 4. SHAP Utils (from shap_utils.py)
# ==========================================
def normalize_shap_values(shap_values, n_features: int) -> np.ndarray:
    if isinstance(shap_values, list): arr = np.stack([np.asarray(v) for v in shap_values], axis=0)
    elif hasattr(shap_values, "values"): arr = np.asarray(shap_values.values)
    else: arr = np.asarray(shap_values)
    if arr.ndim == 1: return np.abs(arr)
    f_axis = [i for i, s in enumerate(arr.shape) if s == n_features][-1]
    r_axes = tuple(i for i in range(arr.ndim) if i != f_axis)
    return np.abs(arr).mean(axis=r_axes)

def compute_tree_shap_summary(model, x_train, selected_features, seed, cfg):
    x_ev = x_train[list(selected_features)].sample(n=min(len(x_train), cfg.tree_shap_sample), random_state=seed)
    try:
        explainer = shap.TreeExplainer(model)
        vals = explainer.shap_values(x_ev)
    except:
        explainer = shap.Explainer(model.predict, x_ev)
        vals = explainer(x_ev)
    mean_abs = normalize_shap_values(vals, len(selected_features))
    return pd.DataFrame({"feature": list(selected_features), "mean_abs_shap": mean_abs}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

def compute_dnn_shap_summary(model, x_train, selected_features, seed, cfg):
    f_list = list(selected_features)
    x_bg = x_train[f_list].sample(n=min(len(x_train), cfg.dnn_bg_sample), random_state=seed).to_numpy(dtype=np.float32)
    x_ev = x_train[f_list].sample(n=min(len(x_train), cfg.dnn_eval_sample), random_state=seed).to_numpy(dtype=np.float32)
    with tf.device(get_tf_device()):
        explainer = shap.GradientExplainer(model, x_bg)
        vals = explainer.shap_values(x_ev)
    mean_abs = normalize_shap_values(vals, len(selected_features))
    return pd.DataFrame({"feature": f_list, "mean_abs_shap": mean_abs}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

def save_shap_outputs(shap_df, repeat, scenario, model_name, exp, k, cfg):
    out = shap_df.copy()
    for i, (col, val) in enumerate(zip(["repeat", "scenario", "model", "exp", "k"], [repeat, scenario, model_name, exp, k])):
        out.insert(i, col, val)
    out["rank"] = np.arange(1, len(out) + 1)
    out.to_csv(os.path.join(cfg.shap_dir, f"{model_name}_{scenario}_{exp}_k{k}_repeat{repeat}_shap.csv"), index=False, encoding="utf-8-sig")
    return out

# ==========================================
# 5. Rand-k Utils (from randk_utils.py)
# ==========================================
def load_recorded_randk(csv_dir: str, supported_models: Sequence[str]):
    records = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
    for path in glob.glob(os.path.join(csv_dir, "*_all_results.csv")):
        m_name = os.path.basename(path).replace("_all_results.csv", "").lower()
        if m_name not in set(supported_models): continue
        df = pd.read_csv(path)
        sub = df[df["exp"].astype(str).str.lower() == "randk"]
        for _, row in sub.iterrows():
            feats = [f.strip() for f in str(row["features"]).split(",") if f.strip()]
            if feats: records[m_name][int(row["repeat"])][int(row["k"])].append(feats)
    return records

# ==========================================
# 6. Result Utils (from result_utils.py)
# ==========================================
def save_metric_outputs(metrics_df: pd.DataFrame, cfg: ExperimentConfig):
    metrics_df.to_csv(os.path.join(cfg.result_dir, "experiment_metrics_all.csv"), index=False, encoding="utf-8-sig")
    summary = metrics_df.groupby(["scenario", "model", "exp", "k", "n_features"], as_index=False).agg(
        param_count_mean=("param_count", "mean"), inference_time_total_ms_mean=("inference_time_total_ms", "mean"),
        inference_time_per_sample_ms_mean=("inference_time_per_sample_ms", "mean")
    )
    summary.to_csv(os.path.join(cfg.result_dir, "experiment_metrics_summary.csv"), index=False, encoding="utf-8-sig")
    return summary

def save_shap_summary_outputs(shap_all_df: pd.DataFrame, cfg: ExperimentConfig):
    if shap_all_df.empty: return pd.DataFrame()
    shap_all_df.to_csv(os.path.join(cfg.result_dir, "experiment_shap_all.csv"), index=False, encoding="utf-8-sig")
    summary = shap_all_df.groupby(["scenario", "model", "exp", "k", "feature"], as_index=False).agg(
        mean_abs_shap_mean=("mean_abs_shap", "mean"), best_rank_mean=("rank", "mean")
    ).sort_values(["scenario", "model", "exp", "k", "mean_abs_shap_mean"], ascending=[True]*4 + [False]).reset_index(drop=True)
    summary.to_csv(os.path.join(cfg.result_dir, "experiment_shap_summary.csv"), index=False, encoding="utf-8-sig")
    return summary

# ==========================================
# 7. Experiment Runner (from experiment_runner.py)
# ==========================================
def run_single_experiment(repeat, scenario, model_name, exp, k, selected_features, x_train, x_test, y_train, y_test, seed, num_classes, cfg):
    print(f"[Run] rep={repeat}, scen={scenario}, model={model_name}, exp={exp}, k={k}")
    if model_name in cfg.ml_models:
        model = train_ml_model(model_name, x_train, y_train, seed)
        params, (inf_t, inf_p) = count_params_ml(model, model_name), measure_inference_time(model, x_test, model_name, cfg)
        shap_df = compute_tree_shap_summary(model, x_train, selected_features, seed, cfg)
    else:
        model = train_dnn_model(x_train, y_train, num_classes, seed, cfg)
        params, (inf_t, inf_p) = count_params_dnn(model), measure_inference_time(model, x_test, "dnn", cfg)
        shap_df = compute_dnn_shap_summary(model, x_train, selected_features, seed, cfg)
    
    shap_out = save_shap_outputs(shap_df, repeat, scenario, model_name, exp, k, cfg)
    metric_row = {
        "repeat": repeat, "scenario": scenario, "model": model_name, "exp": exp, "k": int(k),
        "n_features": len(selected_features), "param_count": params, "inference_time_total_ms": inf_t,
        "inference_time_per_sample_ms": inf_p, "n_train_samples": len(x_train), "n_test_samples": len(x_test),
        "selected_features": ",".join(selected_features)
    }
    return metric_row, shap_out

def run_experiments(target_models, label_encoder, randk_records, cfg: ExperimentConfig):
    all_metrics, all_shaps = [], []
    for repeat in range(1, cfg.n_repeats + 1):
        seed = cfg.global_seed + (repeat - 1)
        print(f"\n[Repeat {repeat}/{cfg.n_repeats}] seed={seed}")
        xt_b_l, xte_b_l, yt_b_l, yte_b_l = load_split_data(cfg.baseline_days, seed, label_encoder, cfg)
        xt_s_l, xte_s_l, yt_s_l, yte_s_l = load_split_data(cfg.select_days, seed, label_encoder, cfg)
        
        b_feats, s_feats = xt_b_l[0].columns.tolist(), xt_s_l[0].columns.tolist()
        xt_b, xte_b, yt_b, yte_b = concat_selected_features(xt_b_l, xte_b_l, yt_b_l, yte_b_l, b_feats)
        xt_s, xte_s, yt_s, yte_s = concat_selected_features(xt_s_l, xte_s_l, yt_s_l, yte_s_l, s_feats)

        rankings = {}
        for m_name in target_models:
            print(f"[SHAP Ranking] model={m_name}")
            if m_name in cfg.ml_models:
                m = train_ml_model(m_name, xt_s, yt_s, seed)
                sdf = compute_tree_shap_summary(m, xt_s, s_feats, seed, cfg)
            else:
                m = train_dnn_model(xt_s, yt_s, len(label_encoder.classes_), seed, cfg)
                sdf = compute_dnn_shap_summary(m, xt_s, s_feats, seed, cfg)
            rankings[m_name] = sdf["feature"].tolist()

        for m_name in target_models:
            # Baseline
            row, s_rows = run_single_experiment(repeat, "baseline_day4_included", m_name, "baseline", len(b_feats), b_feats, xt_b, xte_b, yt_b, yte_b, seed, len(label_encoder.classes_), cfg)
            all_metrics.append(row); all_shaps.append(s_rows)
            
            # Top-k / Bottom-k
            rk = rankings[m_name]
            for k in cfg.k_list:
                for exp_type, f_list in [("topk", rk[:k]), ("bottomk", rk[-k:])]:
                    xt, xte, yt, yte = concat_selected_features(xt_s_l, xte_s_l, yt_s_l, yte_s_l, f_list)
                    row, s_rows = run_single_experiment(repeat, "feature_selected_day4_excluded", m_name, exp_type, k, f_list, xt, xte, yt, yte, seed, len(label_encoder.classes_), cfg)
                    all_metrics.append(row); all_shaps.append(s_rows)
            
            # Rand-k
            for k_val, f_lists in randk_records.get(m_name, {}).get(repeat, {}).items():
                for idx, recorded in enumerate(f_lists, 1):
                    valid = [f for f in recorded if f in s_feats]
                    if not valid: continue
                    xt, xte, yt, yte = concat_selected_features(xt_s_l, xte_s_l, yt_s_l, yte_s_l, valid)
                    row, s_rows = run_single_experiment(repeat, "randk_reproduced_from_csv", m_name, "randk", len(valid), valid, xt, xte, yt, yte, seed, len(label_encoder.classes_), cfg)
                    row["randk_record_index"] = idx; s_rows["randk_record_index"] = idx
                    all_metrics.append(row); all_shaps.append(s_rows)

    return pd.DataFrame(all_metrics), pd.concat(all_shaps, axis=0, ignore_index=True) if all_shaps else pd.DataFrame()

# ==========================================
# 8. Main Execution (from run_experiment.py)
# ==========================================
def main():
    cfg = ExperimentConfig()
    # 주피터 노트북에서 실행 시 아래 경로들을 본인의 환경에 맞게 직접 수정하세요.
    # cfg.data_dir = r"실제 데이터 경로"
    
    cfg.ensure_output_dirs()
    configure_tensorflow_gpu()
    
    # 실행할 모델 선택 (예: ["dt", "rf", "dnn"] 또는 ["all"])
    target_models = cfg.all_models 
    
    le = build_global_label_encoder(sorted(set(cfg.baseline_days + cfg.select_days)), cfg)
    randk_records = load_recorded_randk(cfg.prev_result_csv_dir, cfg.all_models)
    
    metrics_df, shap_all_df = run_experiments(target_models, le, randk_records, cfg)
    
    save_metric_outputs(metrics_df, cfg)
    save_shap_summary_outputs(shap_all_df, cfg)
    
    print(f"\n[Done] Results saved to: {cfg.result_dir}")

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'catboost'

In [ ]:
!pip install catboost